# DNN Lab 03: Regularization Techniques & Hyperparameter Tuning

### Dataset format

The CSV file should contain:

`label, pixel_0, pixel_1, ..., pixel_783`

There are 785 columns in total:
- 1 target column: `label`
- 784 input features: `pixel_0` to `pixel_783`

> **Important:** The test set is used only once for final evaluation. Hyperparameters are selected using the validation set.

## 1. Imports

In [ ]:
import os
import random
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

print("PyTorch version:", torch.__version__)

#### GPU Configuration

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("CUDA is not available. Training will use CPU.")

## 2. Load the MNIST CSV

Change `CSV_PATH` to the location/name of your dataset.

Example:

```python
CSV_PATH = "mnist.csv"
```

If your file has a different name, change it here only.

In [ ]:
CSV_PATH = "train.csv"   #

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)
display(df.head())

#### Inspect the Dataset

In [ ]:
print("Columns:", len(df.columns))
print("First columns:", df.columns[:10].tolist())
print("Last columns:", df.columns[-5:].tolist())

print("\nMissing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nLabel distribution:")
display(df["label"].value_counts().sort_index())

#### Separate Features and Labels

The first column is the target:

- `label` → digit class 0–9
- `pixel_0 ... pixel_783` → 784 input features

In [ ]:
X = df.drop(columns=["label"]).values
y = df["label"].values

print("X shape:", X.shape)
print("y shape:", y.shape)

assert X.shape[1] == 784, "Expected exactly 784 pixel features."

#### Check Pixel Range

In [ ]:
print("Minimum pixel value:", X.min())
print("Maximum pixel value:", X.max())

#### Visualize Some Original Digits

Each row contains 784 pixels. We reshape the 784 values into:

`28 × 28`

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))

for i, ax in enumerate(axes.flat):
    ax.imshow(X[i].reshape(28, 28), cmap="gray")
    ax.set_title(f"Label: {y[i]}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 3. Train / Validation / Test Split

We use:

- **80% training**
- **10% validation**
- **10% test**

`stratify=y` keeps the class distribution approximately balanced across splits.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.40,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

## 4. Normalize Pixel Values

MNIST pixels are usually in `[0, 255]`.

We scale them to `[0, 1]`:

\[
x_{normalized} = \frac{x}{255}
\]

This generally makes neural-network optimization more stable.

In [ ]:
X_train = X_train.astype(np.float32) / 255.0
X_val   = X_val.astype(np.float32) / 255.0
X_test  = X_test.astype(np.float32) / 255.0

print("Train range:", X_train.min(), "to", X_train.max())

## 4. PyTorch Dataset

In [ ]:
class MNISTCSVDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]


train_dataset = MNISTCSVDataset(X_train, y_train)
val_dataset   = MNISTCSVDataset(X_val, y_val)
test_dataset  = MNISTCSVDataset(X_test, y_test)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

In [ ]:
train_dataset[0]

#### DataLoaders

`pin_memory=True` can improve host-to-GPU transfer when CUDA is being used.

`non_blocking=True` will be used during tensor transfer to the GPU.

In [ ]:
BATCH_SIZE = 128

loader_kwargs = {
    "num_workers": 0,
    "pin_memory": torch.cuda.is_available()
}

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    **loader_kwargs
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_kwargs
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_kwargs
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

#### Inspect One Batch

In [ ]:
X_batch, y_batch = next(iter(train_loader))

print("Input batch shape:", X_batch.shape)
print("Label batch shape:", y_batch.shape)
print("Input dtype:", X_batch.dtype)
print("Label dtype:", y_batch.dtype)

## 5. PyTorch DNN Modeling

# Part A: Baseline DNN

We first create a model **without explicit regularization**.

Architecture:

```text
784
 ↓
512
 ↓
256
 ↓
128
 ↓
10
```

The purpose is to establish a baseline before adding regularization.

In [ ]:
class BaselineMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU(),

            nn.Linear(512, 256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)


baseline_model = BaselineMLP().to(device)

print(baseline_model)
from torchinfo import summary
summary(baseline_model)

### Optimized Training and Evaluation Functions

Important steps:

- `model.train()` → training mode
- `model.eval()` → evaluation mode
- `torch.no_grad()` → no gradient calculation during validation
- `zero_grad(set_to_none=True)` → efficient gradient reset
- `non_blocking=True` → efficient CPU → GPU transfer when pinned memory is enabled

In [ ]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
    regularization=None,
    reg_lambda=0.0
):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(
            device,
            non_blocking=True
        )
        y_batch = y_batch.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        if regularization == "l1":
            l1 = sum(
                p.abs().sum()
                for p in model.parameters()
                if p.requires_grad
            )
            loss = loss + reg_lambda * l1

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(
            device,
            non_blocking=True
        )
        y_batch = y_batch.to(
            device,
            non_blocking=True
        )

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        running_loss += loss.item() * X_batch.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

    loss = running_loss / total
    accuracy = correct / total

    return loss, accuracy

### General Training Function

This function will be reused for every regularization experiment.

That is important scientifically: **we keep the training pipeline consistent and change only the regularization method.**

In [ ]:
def fit_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs=10,
    regularization=None,
    reg_lambda=0.0,
    early_stopping=False,
    patience=3,
    verbose=True
):

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device,
            regularization=regularization,
            reg_lambda=reg_lambda
        )

        val_loss, val_acc = evaluate(
            model,
            val_loader,
            criterion,
            device
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if verbose:
            print(
                f"Epoch {epoch+1:02d}/{epochs} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Train Acc: {train_acc:.4f} | "
                f"Val Loss: {val_loss:.4f} | "
                f"Val Acc: {val_acc:.4f}"
            )

        if early_stopping:

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                patience_counter += 1

            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}.")
                break

    if early_stopping and best_state is not None:
        model.load_state_dict(best_state)

    return history

### Plot Training History

In [ ]:
def plot_history(history, title):

    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(7, 4))
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title + " — Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(epochs, history["train_acc"], label="Train Accuracy")
    plt.plot(epochs, history["val_acc"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title + " — Accuracy")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# Experiment 1: Baseline

No L1, no weight decay, no Dropout, no BatchNorm, and no early stopping.

This gives us a reference point.

In [ ]:
EPOCHS = 15
LEARNING_RATE = 1e-3

model_baseline = BaselineMLP().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model_baseline.parameters(),
    lr=LEARNING_RATE
)

start = time.time()

history_baseline = fit_model(
    model_baseline,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs=EPOCHS
)

print(f"Training time: {time.time() - start:.2f} seconds")

In [ ]:
plot_history(history_baseline, "Baseline MLP")

## Baseline Observation

Look at the gap between:

- training accuracy
- validation accuracy

A large and increasing gap is evidence of overfitting.

**Question:** At which epoch does the model begin showing signs of overfitting?

# Part B: L1 Regularization

L1 adds a penalty:

$$
L_{total} = L_{CE} + \lambda \sum_i |w_i|
$$

L1 tends to encourage small or zero-valued weights.

We use the same model architecture.

In [ ]:
L1_LAMBDA = 1e-6

model_l1 = BaselineMLP().to(device)

optimizer_l1 = torch.optim.Adam(
    model_l1.parameters(),
    lr=LEARNING_RATE
)

history_l1 = fit_model(
    model_l1,
    train_loader,
    val_loader,
    criterion,
    optimizer_l1,
    device,
    epochs=EPOCHS,
    regularization="l1",
    reg_lambda=L1_LAMBDA
)

In [ ]:
plot_history(history_l1, "L1 Regularization")

### L1 Experiment Questions

1. Did training accuracy decrease?
2. Did the train/validation gap change?
3. What happens if `L1_LAMBDA` is increased?

# Part C: L2 Regularization / Weight Decay

A common practical way to apply L2-style regularization is through optimizer `weight_decay`.

Conceptually:

$$
L_{total} = L_{CE} + \lambda \sum_i w_i^2
$$

In [ ]:
WEIGHT_DECAY = 1e-4

model_l2 = BaselineMLP().to(device)

optimizer_l2 = torch.optim.AdamW(
    model_l2.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

history_l2 = fit_model(
    model_l2,
    train_loader,
    val_loader,
    criterion,
    optimizer_l2,
    device,
    epochs=EPOCHS
)

In [ ]:
plot_history(history_l2, "L2 / Weight Decay")

### Why AdamW?

For this lab we use `AdamW`, which implements decoupled weight decay. This lets students see weight decay as a hyperparameter without manually adding an L2 term to the loss.

**Question:** What happens if `weight_decay` is increased from `1e-4` to `1e-3`?

# Part D: Dropout

Dropout randomly disables units during training.

Example:

```text
Before Dropout:

● ● ● ● ● ● ●

After Dropout:

● ○ ● ● ○ ● ○
```

The model therefore cannot depend too heavily on a small set of neurons.

In [ ]:
class DropoutMLP(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)


DROPOUT_RATE = 0.3

model_dropout = DropoutMLP(DROPOUT_RATE).to(device)

optimizer_dropout = torch.optim.Adam(
    model_dropout.parameters(),
    lr=LEARNING_RATE
)

history_dropout = fit_model(
    model_dropout,
    train_loader,
    val_loader,
    criterion,
    optimizer_dropout,
    device,
    epochs=EPOCHS
)

In [ ]:
plot_history(history_dropout, f"Dropout = {DROPOUT_RATE}")

### Dropout Experiment

Try:

```python
DROPOUT_RATE = 0.1
DROPOUT_RATE = 0.3
DROPOUT_RATE = 0.5
```

Compare validation accuracy.

Remember: very strong regularization can also cause underfitting.

# Part E: Batch Normalization

BatchNorm normalizes intermediate activations during training
For your DNN lab, I would present it as:

$$ \boxed{z=Wx+b} $$

↓

Batch mean
$$ \boxed{\mu_B=\frac{1}{m}\sum_{i=1}^{m}z_i} $$

↓

Batch variance
$$ \boxed{ \sigma_B^2= \frac{1}{m}\sum_{i=1}^{m}(z_i-\mu_B)^2 } $$

↓

Normalize
$$ \boxed{ \hat z_i= \frac{z_i-\mu_B} {\sqrt{\sigma_B^2+\epsilon}} } $$

↓

Scale and shift
$$ \boxed{ y_i=\gamma\hat z_i+\beta } $$

↓

Activation
$$ \boxed{ a_i=\max(0,y_i) } $$

So the complete pipeline is:

$$ \boxed{ x \rightarrow Wx+b \rightarrow BN \rightarrow ReLU \rightarrow a } $$

Then gradient descent updates \(\gamma\):

$$ \boxed{ \gamma_{\text{new}} = \gamma_{\text{old}} - \eta \frac{\partial L}{\partial\gamma} } $$

where \(\eta\) is the learning rate.

And gradient descent Beta:

$$ \boxed{ \beta_{\text{new}} = \beta_{\text{old}} - \eta \frac{\partial L}{\partial\beta} } $$

It is **not identical to Dropout or L2**, although it can have a regularizing effect.

In [ ]:
class BatchNormMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(784, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)


model_bn = BatchNormMLP().to(device)
print("BatchNorm \n",model_bn)

print("\nTrainable parameters:",

      sum(p.numel() for p in model_bn.parameters() if p.requires_grad))

                 BATCH NORMALIZATION
                         │
          ┌──────────────┴──────────────┐
          │                             │
    Learnable parameters          Running statistics
          │                             │
      γ and β                       mean & variance
          │                             │
   
   Backpropagation                  Batch statistics
   
          │                             │
   
     Gradients                    momentum = 0.1
          
          │                             │
   Learning rate                        
          
          │                             │
          
          ▼                             ▼
     
     γ and β update             running stats update

     

In [ ]:
from torchinfo import summary
summary(model_bn)

In [ ]:
optimizer_bn = torch.optim.Adam(
    model_bn.parameters(),
    lr=LEARNING_RATE
)

history_bn = fit_model(
    model_bn,
    train_loader,
    val_loader,
    criterion,
    optimizer_bn,
    device,
    epochs=EPOCHS
)

In [ ]:
plot_history(history_bn, "Batch Normalization")

# Part F: Early Stopping

Early stopping monitors validation performance.

If validation loss does not improve for a specified number of epochs (`patience`), training stops.

The best model state is restored.

In [ ]:
model_early = BaselineMLP().to(device)

optimizer_early = torch.optim.Adam(
    model_early.parameters(),
    lr=LEARNING_RATE
)

history_early = fit_model(
    model_early,
    train_loader,
    val_loader,
    criterion,
    optimizer_early,
    device,
    epochs=30,
    early_stopping=True,
    patience=3
)

In [ ]:
plot_history(history_early, "Early Stopping")

# Part G: Combine Regularization Techniques

Now we create a more flexible model containing:

- BatchNorm
- Dropout
- L2 / weight decay
- Early stopping

In [ ]:
class RegularizedMLP(nn.Module):
    def __init__(self, hidden_size=512, dropout=0.3):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(784, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_size, hidden_size // 2),
            nn.BatchNorm1d(hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_size // 2, 128),
            nn.ReLU(),

            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
model_combined = RegularizedMLP(
    hidden_size=512,
    dropout=0.3
).to(device)

optimizer_combined = torch.optim.AdamW(
    model_combined.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

history_combined = fit_model(
    model_combined,
    train_loader,
    val_loader,
    criterion,
    optimizer_combined,
    device,
    epochs=30,
    early_stopping=True,
    patience=4
)

In [ ]:
plot_history(history_combined, "Combined Regularization")

# Part H: Compare Regularization Experiments

We compare the best validation accuracy observed by each experiment.

> These results depend on your dataset, random seed, GPU, PyTorch version, and hyperparameters. Do not assume one method will always produce the highest score.

In [ ]:
def best_val_accuracy(history):
    return max(history["val_acc"])


comparison = pd.DataFrame({
    "Experiment": [
        "Baseline",
        "L1",
        "L2 / Weight Decay",
        "Dropout",
        "BatchNorm",
        "Early Stopping",
        "Combined"
    ],
    "Best Validation Accuracy": [
        best_val_accuracy(history_baseline),
        best_val_accuracy(history_l1),
        best_val_accuracy(history_l2),
        best_val_accuracy(history_dropout),
        best_val_accuracy(history_bn),
        best_val_accuracy(history_early),
        best_val_accuracy(history_combined)
    ]
})

display(
    comparison.sort_values(
        "Best Validation Accuracy",
        ascending=False
    ).reset_index(drop=True)
)

# Part I: Final Test Evaluation

Choose between different model we trained above

In [ ]:
test_loss, test_acc = evaluate(
    model_combined,
    test_loader,
    criterion,
    device
)

print(f"Final Test Loss: {test_loss:.4f}")
print(f"Final Test Accuracy: {test_acc:.4f}")

## 6. Confusion Matrix

In [ ]:
@torch.no_grad()
def get_predictions(model, loader, device):

    model.eval()

    all_predictions = []
    all_labels = []

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(
            device,
            non_blocking=True
        )

        outputs = model(X_batch)

        predictions = outputs.argmax(dim=1).cpu().numpy()

        all_predictions.extend(predictions)
        all_labels.extend(y_batch.numpy())

    return np.array(all_labels), np.array(all_predictions)


y_true, y_pred = get_predictions(
    model_combined,
    test_loader,
    device
)

cm = confusion_matrix(
    y_true,
    y_pred
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=np.arange(10)
)

fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Final Model — MNIST Confusion Matrix")
plt.show()

## 7. Classification Report

In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        digits=4
    )
)

# Part N: Inspect Incorrect Predictions


Let's inspect examples that the model classified incorrectly.

In [ ]:
wrong_indices = np.where(y_true != y_pred)[0]

print("Incorrect predictions:", len(wrong_indices))
print("Total test samples:", len(y_true))

In [ ]:
n_show = min(20, len(wrong_indices))

fig, axes = plt.subplots(
    4,
    5,
    figsize=(10, 8)
)

for ax, idx in zip(axes.flat, wrong_indices[:n_show]):

    image = X_test[idx].reshape(28, 28)

    ax.imshow(image, cmap="gray")
    ax.set_title(
        f"True: {y_true[idx]} | Pred: {y_pred[idx]}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

# Final Experimental Comparison

| Method | What changed? | Effect on training | Effect on validation |
|---|---|---|---|
| Baseline | No regularization | | |
| L1 | Absolute-weight penalty | | |
| L2 | Weight decay | | |
| Dropout | Random neuron deactivation | | |
| BatchNorm | Activation normalization | | |
| Early stopping | Stops based on validation loss | | |
| Combined | Multiple techniques | | |